# CoT Synthesis - FULL RUN (Kaggle GPU / vLLM)

Teacher distillation skala penuh pakai **vLLM lokal** (tanpa limit API).

```
merged_dataset*.jsonl
   -> dedup (merge_and_dedup, by soal)         data/train_pool.jsonl
   -> generate  (teacher vLLM, N solusi/soal)  cot/candidates.jsonl
   -> filter    (judge vLLM, buang yg salah)   cot/correct.jsonl
   -> to_chatml (cot + nocot)                   sft/{cot,nocot}.jsonl
```

**Prasyarat Kaggle:**
1. Settings -> Accelerator = **GPU T4 x2**.
2. Settings -> Internet = **ON** (buat clone repo + download model HF).
3. Upload `merged_dataset.jsonl` (dan/atau `merged_dataset_un_cleanvlm_aimo5000.jsonl`) sebagai **Kaggle Dataset**, lalu Add ke notebook.

Full 25k makan waktu lama. Buat coba dulu set `LIMIT` kecil. Buat full: `LIMIT=None`, dan kalau sesi habis, jalankan lagi (resumable lewat `candidates.jsonl`).

## 0. Setup (install + clone repo)

In [ ]:
!pip install -q vllm math-verify

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = '/kaggle/working/FP_NLP'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/henray404/FP_NLP.git', REPO], check=True)
sys.path.insert(0, REPO)
WORK = Path('/kaggle/working')
print('repo:', REPO)

## 1. Cari data (Kaggle Dataset yang di-Add)

In [ ]:
INPUTS = [str(p) for p in Path('/kaggle/input').rglob('merged_dataset*.jsonl')]
assert INPUTS, 'Tidak ketemu. Upload merged_dataset*.jsonl sebagai Kaggle Dataset lalu Add ke notebook.'
print('input files:')
for p in INPUTS:
    print(' -', p)

## 2. Dedup + buang soal tanpa jawaban

Dedup by `soal` (simpan versi paling lengkap). Lalu buang baris tanpa `jawaban` (judge butuh gold answer).

In [ ]:
import json
from src.preprop.merge_and_dedup import run as merge_dedup

DEDUP = WORK / 'dataset_dedup.jsonl'
print(merge_dedup([Path(p) for p in INPUTS], DEDUP))

POOL = WORK / 'train_pool.jsonl'
n = 0
with open(DEDUP, encoding='utf-8') as f, open(POOL, 'w', encoding='utf-8') as o:
    for line in f:
        d = json.loads(line)
        if (d.get('jawaban') or '').strip():
            o.write(json.dumps(d, ensure_ascii=False) + '\n')
            n += 1
print('train_pool (ada jawaban):', n, '->', POOL)

## 3. Config

In [ ]:
TEACHER_MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'  # teacher (reasoning)
JUDGE_MODEL   = 'Qwen/Qwen2.5-7B-Instruct'                # judge

N_CANDIDATES = 4       # solusi per soal
MAX_TOKENS   = 4096    # R1 reasoning panjang; GPU jadi gak ada limit TPM
TP           = 2       # 2x T4 -> tensor parallel 2
LIMIT        = 50      # COBA dulu. Full: None

COT_DIR = WORK / 'cot'
SFT_DIR = WORK / 'sft'
CAND    = COT_DIR / 'candidates.jsonl'
CORRECT = COT_DIR / 'correct.jsonl'
print('teacher:', TEACHER_MODEL, '| N:', N_CANDIDATES, '| limit:', LIMIT)

## 4. Generate (teacher, vLLM)

Resumable: kalau sesi putus, jalankan lagi — soal yang sudah `N` kandidat dilewati.

In [ ]:
from src.cot_synthesis.generate import run_generate

gen_stats = run_generate(
    POOL, CAND, backend='vllm', model=TEACHER_MODEL,
    n=N_CANDIDATES, max_tokens=MAX_TOKENS, limit=LIMIT, tensor_parallel_size=TP,
)
print(gen_stats)

## 5. Bebaskan GPU sebelum load judge

Kalau cell filter di bawah **OOM**: Restart kernel, lalu jalankan ulang cell Setup -> Config -> Filter (file `candidates.jsonl` di /kaggle/working tetap ada).

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print('GPU dibersihkan')

## 6. Filter (judge, vLLM)

In [ ]:
from src.cot_synthesis.filter_solutions import run_filter

flt_stats = run_filter(
    CAND, CORRECT, judge_backend='vllm', judge_model=JUDGE_MODEL, tensor_parallel_size=TP,
)
acc = flt_stats['kept'] / max(flt_stats['total'], 1)
print(flt_stats)
print(f'acceptance rate: {acc:.1%}')

## 7. Build ChatML (cot + nocot)

In [ ]:
from src.cot_synthesis.to_chatml import run as build_chatml

ml_stats = build_chatml(CORRECT, SFT_DIR, max_per_problem=4, max_solution_chars=8000)
print(ml_stats)

## 8. Download hasil

Output ada di `/kaggle/working/sft/` dan `/kaggle/working/cot/`. Zip biar gampang diunduh, atau pakai **Save Version** biar persist antar sesi.

In [ ]:
import shutil
from IPython.display import FileLink

out_zip = '/kaggle/working/cot_sft_outputs'
# kumpulin cot/ + sft/ ke satu folder lalu zip
bundle = WORK / 'bundle'
bundle.mkdir(exist_ok=True)
for d in ['cot', 'sft']:
    src = WORK / d
    if src.exists():
        shutil.copytree(src, bundle / d, dirs_exist_ok=True)
shutil.make_archive(out_zip, 'zip', str(bundle))
print('zip:', out_zip + '.zip')
FileLink('cot_sft_outputs.zip')